In [ ]:
import pandas as pd
from textblob import TextBlob
import ipywidgets as widgets
from IPython.display import display, clear_output
import io
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# --- 1. LÓGICA DE ANÁLISIS (TextBlob) ---
def analizar_sentimiento(texto):
    if pd.isna(texto): 
        return "Neutral"
    # TextBlob analiza la polaridad: -1 (negativo) a 1 (positivo)
    analisis = TextBlob(str(texto))
    polaridad = analisis.sentiment.polarity
    
    if polaridad > 0.1:
        return 'Positivo'
    elif polaridad < -0.1:
        return 'Negativo'
    else:
        return 'Neutral'

# --- 2. COMPONENTES DE LA INTERFAZ ---
uploader = widgets.FileUpload(
    accept='.xlsx', 
    multiple=False, 
    description="Cargar Excel"
)
boton_procesar = widgets.Button(
    description="Generar Dashboard", 
    button_style='success', 
    icon='area-chart',
    layout=widgets.Layout(width='200px', height='40px')
)
salida_dashboard = widgets.Output()

# --- 3. FUNCIÓN PRINCIPAL DEL BOTÓN ---
def generar_dashboard(b):
    with salida_dashboard:
        clear_output()
        
        # Validar si hay archivo
        if not uploader.value:
            print("❌ Por favor, selecciona un archivo .xlsx primero.")
            return
        
        print("⏳ Procesando datos en Google Cloud... un momento.")
        
        # Obtener el archivo (Soporte para JupyterLab 4/Tupla)
        input_file = uploader.value[0]
        contenido = input_file['content']
        
        try:
            # Leer Excel
            df = pd.read_excel(io.BytesIO(contenido))
            
            # Buscar la columna 'Comentario' (flexible con mayúsculas/minúsculas)
            col_name = next((c for c in df.columns if c.lower() == 'comentario'), None)
            
            if col_name:
                # Aplicar análisis
                df['Sentimiento'] = df[col_name].apply(analizar_sentimiento)
                
                # --- CONFIGURACIÓN VISUAL DEL DASHBOARD ---
                fig = plt.figure(figsize=(16, 12))
                grid = plt.GridSpec(2, 2, wspace=0.3, hspace=0.4)
                colores = {'Positivo': '#2ecc71', 'Neutral': '#f1c40f', 'Negativo': '#e74c3c'}
                
                # A. Gráfico de Barras (Corregido para evitar el Future Warning)
                ax_bar = fig.add_subplot(grid[0, 0])
                sns.countplot(
                    x='Sentimiento', 
                    data=df, 
                    ax=ax_bar, 
                    hue='Sentimiento', 
                    palette=colores, 
                    order=['Positivo', 'Neutral', 'Negativo'],
                    legend=False
                )
                ax_bar.set_title('Conteo Total de Sentimientos', fontsize=14)
                
                # B. Gráfico de Pastel
                ax_pie = fig.add_subplot(grid[0, 1])
                counts = df['Sentimiento'].value_counts()
                counts.plot.pie(
                    autopct='%1.1f%%', 
                    ax=ax_pie, 
                    colors=[colores.get(x, '#ccc') for x in counts.index],
                    textprops={'fontsize': 12}
                )
                ax_pie.set_title('Distribución Porcentual', fontsize=14)
                ax_pie.set_ylabel('')
                
                # C. Nube de Palabras
                ax_word = fig.add_subplot(grid[1, :])
                texto_unido = " ".join(df[col_name].astype(str))
                wordcloud = WordCloud(
                    width=1000, 
                    height=400, 
                    background_color='white', 
                    colormap='viridis',
                    max_words=100
                ).generate(texto_unido)
                ax_word.imshow(wordcloud, interpolation='bilinear')
                ax_word.axis('off')
                ax_word.set_title('Palabras más repetidas en los comentarios', fontsize=14)

                plt.show()
                
                # Opcional: Guardar una copia analizada en la carpeta de GCP
                df.to_excel('ultimo_analisis_realizado.xlsx', index=False)
                print("✅ Dashboard generado con éxito.")
                print("📂 También se ha guardado 'ultimo_analisis_realizado.xlsx' en tu carpeta izquierda.")
                
            else:
                print(f"❌ Error: No encontré la columna 'Comentario'. Columnas disponibles: {list(df.columns)}")
        
        except Exception as e:
            print(f"❌ Ocurrió un error inesperado: {e}")

# --- 4. LANZAR INTERFAZ ---
boton_procesar.on_click(generar_dashboard)

print("        --- ANALIZADOR DE SENTIMIENTOS CRISTIAN ---")
print("1. Sube tu archivo Excel.")
print("2. Asegúrate de que la columna se llame 'Comentario'.")
print("3. Presiona el botón verde.\n")
display(uploader)
display(boton_procesar)
display(salida_dashboard)